# 02 — Sổ Tay Che Lọc PII, Bóc Từ Khóa (`KW_01..KW_16`) & Phân Cụm Du Khách (`C1–C4`)

**Ánh xạ trực tiếp từ thiết kế `2026-11-20-traveler-keyword-clustering-design.md`:**
1. **Bước 0 — PII Scrubbing bắt buộc:** Xóa sạch SĐT, email, CCCD/hộ chiếu, URL, tên riêng khách **trước** khi văn bản đi qua OpenAI API.
2. **Tầng 1 (0đ) + Tầng 2 (OpenAI Structured / Batch API):** Bóc 16 mã `KW_01..KW_16` đa ngôn ngữ (`VI · EN · KO`), tôn trọng luật phủ định 4 từ.
3. **Giai đoạn 1 (Chấm điểm trọng số `C1–C4`) vs. Giai đoạn 2 (`K-Means` + `Silhouette Score`):** Bàn cân thử nghiệm chuẩn bị cho lộ trình tự huấn luyện ML nội bộ giảm phụ thuộc OpenAI API.

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
NOTEBOOK_ROOT = next(
    (p for p in (cwd, cwd.parent, cwd / "analytics-notebooks") if (p / "src").is_dir()),
    cwd,
)
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from src.ai_provider import build_batch_jsonl_lines, create_ai_provider
from src.bq_client import GoodNstayAnalyticsLoader
from src.config import load_settings
from src.keyword_rules import extract_tier1_keywords, score_and_assign_cluster
from src.pii_scrubber import scrub_pii

settings = load_settings()
loader = GoodNstayAnalyticsLoader(settings)
provider = create_ai_provider(settings)
print(f"Active AI provider: {provider.provider_name} ({provider.model_used}) | Key: {settings.masked_openai_key()}")

## 1. Kiểm Định Che Lọc PII (`0% PII Leak`) Trước Khi Gọi OpenAI API

In [ ]:
df_bids = loader.load_bid_telemetry_sample()

sample_pii_cases = [
    {
        "case": "VI_contact_and_baby_cot",
        "name": "Minh Anh",
        "raw": "Minh Anh đặt ngày 2026-11-06, cần nôi em bé và không mùi thuốc lá, gọi +84 905 412 819 hoặc minh.anh@example.com",
    },
    {
        "case": "VI_negation_breakfast",
        "name": "Hoang Nam",
        "raw": "Không cần bữa sáng nhưng cần nhận phòng sớm và xuất hóa đơn VAT, CCCD 079201001234",
    },
    {
        "case": "KO_family_ocean",
        "name": "Kim Ji-hoon",
        "raw": "아이와 가족여행, 조식포함 오션뷰 원해요. 연락처 010-2345-6789",
    },
]

audit_rows = []
for item in sample_pii_cases:
    scrubbed = scrub_pii(item["raw"], known_names=[item["name"]])
    t1 = extract_tier1_keywords(item["raw"], known_names=[item["name"]])
    audit_rows.append(
        {
            "case": item["case"],
            "had_pii": scrubbed.had_pii,
            "cleaned_text": scrubbed.cleaned_text,
            "tier1_keywords": list(t1.matched_keywords),
            "route_to_tier2_llm": t1.needs_tier2_llm,
        }
    )

pd.DataFrame(audit_rows)

## 2. Chấm Điểm Phân Cụm Giai Đoạn 1 (`C1–C4`) & Chuẩn Bị OpenAI Batch JSONL (`T-1`)

In [ ]:
cluster_rows = []
tier2_candidates = []

for row in df_bids.to_dict(orient="records"):
    combined_note = f"{row.get('guest_note_vi', '')} | {row.get('guest_note_en', '')}"
    t1 = extract_tier1_keywords(combined_note, known_names=[row.get("guest_full_name", "")])
    if t1.needs_tier2_llm:
        tier2_candidates.append({"custom_id": str(row["request_code"]), "text": t1.cleaned_text})

    assigned = score_and_assign_cluster(row, note_keywords=t1.matched_keywords)
    cluster_rows.append(
        {
            "request_code": row["request_code"],
            "branch_id": row["branch_id"],
            "status": row["status"],
            "cluster": assigned.cluster,
            "cluster_label_vi": assigned.cluster_label_vi,
            "matched_keywords": list(assigned.matched_note_keywords),
        }
    )

df_clusters = pd.DataFrame(cluster_rows)
df_clusters["is_won"] = (df_clusters["status"] == "accepted").astype(int)

batch_lines = build_batch_jsonl_lines(tier2_candidates[:10], model=settings.openai_model)
print(f"Tổng yêu cầu: {len(df_clusters)} | Chuyển Tầng 2 LLM: {len(tier2_candidates)} | Preview Batch JSONL: {len(batch_lines)} dòng")

cluster_summary = (
    df_clusters.groupby(["cluster", "cluster_label_vi"], as_index=False)
    .agg(requests=("request_code", "count"), won=("is_won", "sum"))
)
cluster_summary["share_pct"] = (cluster_summary["requests"] / max(len(df_clusters), 1) * 100.0).round(1)
cluster_summary["win_rate_pct"] = (cluster_summary["won"] / cluster_summary["requests"].clip(lower=1) * 100.0).round(1)
cluster_summary

## 3. Thử Nghiệm Giai Đoạn 2: Máy Học Không Giám Sát (`K-Means` & `Silhouette Score`)
Kiểm định chất lượng phân cụm trên 6 đặc trưng hành vi liên tục (ngưỡng kích hoạt chuẩn thiết kế: `Silhouette Score >= 0.25` trên >= 5.000 đơn sạch).

In [ ]:
feature_df = pd.DataFrame(
    {
        "log_lead_time": np.log1p(df_bids["lead_time_days"].clip(lower=0)),
        "nights": df_bids["nights"].clip(lower=1),
        "total_guests": df_bids["total_guests"].clip(lower=1),
        "criteria_count": df_bids["criteria_count"].clip(lower=0),
        "max_scroll_depth_pct": df_bids["max_scroll_depth_pct"].clip(lower=0),
        "log_total_dwell_ms": np.log1p(df_bids["total_dwell_ms"].clip(lower=0)),
    }
)
X_scaled = StandardScaler().fit_transform(feature_df)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)
sil = round(float(silhouette_score(X_scaled, labels)), 4)
sizes = pd.Series(labels).value_counts().sort_index().to_dict()
print(f"Prototype K-Means (k=4) Silhouette Score: {sil} (ngưỡng chuẩn >= 0.25) | Cluster sizes: {sizes}")